# MedVQA Fine-Tuning: LLaVA-Med on Custom Chest X-Ray Dataset
**Parth Lathiya (23BCE215) | Dharm Patel (23BCE224) | Nirma University**

## Dataset Info
- `medvqa_dataset.csv` → 218,652 QA pairs (CheXpert + NIH)
- `VQA_RAD Dataset Public.json` → 2,248 original VQA-RAD pairs
- 6 question types: presence, normal_abnormal, identification, multiple_choice, body_system, two_findings
- Model: `microsoft/llava-med-v1.5-mistral-7b` with LoRA fine-tuning

## Cell 1: Install Dependencies

In [ ]:
import subprocess, sys

packages = [
    'transformers==4.40.0',
    'peft==0.10.0',
    'bitsandbytes==0.43.1',
    'accelerate==0.29.3',
    'trl==0.8.6',
    'einops',
    'sentencepiece',
    'Pillow',
    'pandas',
    'scikit-learn',
    'nltk',
    'evaluate',
    'rouge_score',
]

for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])

print('All packages installed successfully.')

## Cell 2: Imports & GPU Check

In [ ]:
import os
import json
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image
import torch
import nltk
nltk.download('punkt', quiet=True)

# GPU info
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f'GPU {i}: {props.name}  VRAM: {props.total_memory / 1e9:.1f} GB')

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

## Cell 3: Configure Paths & Hyperparameters

In [ ]:
# ─────────────────────────────────────────────
# CONFIGURE THESE PATHS FOR YOUR KAGGLE SETUP
# ─────────────────────────────────────────────

# Path to your uploaded medvqa_dataset.csv
CSV_PATH = '/kaggle/input/medvqa-custom-dataset/medvqa_dataset.csv'

# VQA-RAD JSON (for additional chest-only pairs)
VQARAD_JSON = '/kaggle/input/medvqa-custom-dataset/VQA_RAD Dataset Public.json'

# VQA-RAD images folder
VQARAD_IMG_DIR = '/kaggle/input/medvqa-custom-dataset/VQA_RAD Image Folder'

# CheXpert images root (Kaggle dataset: ashery/chexpert)
CHEXPERT_ROOT = '/kaggle/input/chexpert'

# NIH images root (Kaggle dataset: nih-chest-xrays)
NIH_ROOT = '/kaggle/input/nih-chest-xrays'

# LLaVA-Med HuggingFace model name
MODEL_ID = 'microsoft/llava-med-v1.5-mistral-7b'

# Output directory for saved model
OUTPUT_DIR = '/kaggle/working/medvqa_adapter'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ─────────────────────────────────────────────
# TRAINING HYPERPARAMETERS
# ─────────────────────────────────────────────
CFG = {
    # LoRA
    'lora_r'          : 8,
    'lora_alpha'      : 16,
    'lora_dropout'    : 0.05,
    'target_modules'  : ['q_proj', 'v_proj', 'k_proj', 'o_proj'],

    # Training
    'learning_rate'   : 2e-4,
    'batch_size'      : 4,         # per GPU
    'grad_accum'      : 8,         # effective batch = 32
    'epochs'          : 3,
    'warmup_ratio'    : 0.03,
    'max_seq_length'  : 512,
    'image_size'      : 224,

    # Data sampling (set to None to use ALL training data)
    # For quick test use 5000, for full training use None
    'max_train_samples': 40000,
    'max_val_samples'  : 5000,

    # Seed
    'seed': 42,
}

random.seed(CFG['seed'])
np.random.seed(CFG['seed'])
torch.manual_seed(CFG['seed'])

print('Configuration set.')
print(json.dumps(CFG, indent=2))

## Cell 4: Load & Preprocess Dataset

In [ ]:
print('Loading medvqa_dataset.csv ...')
df = pd.read_csv(CSV_PATH)
print(f'Total rows: {len(df):,}')
print(df.groupby(['split', 'answer_type']).size().to_string())

# ─────────────────────────────────────────────
# FIX IMAGE PATHS for Kaggle environment
# The CSV has paths like:
#   /kaggle/input/datasets/ashery/chexpert/train/...
#   /kaggle/input/datasets/nih-chest-xrays/data/...
# We remap them to actual Kaggle input paths
# ─────────────────────────────────────────────

def fix_image_path(path: str) -> str:
    """Remap dataset paths to actual Kaggle paths."""
    if 'chexpert' in path.lower() or 'ashery' in path.lower():
        # Extract relative part after 'chexpert/'
        idx = path.lower().find('chexpert/')
        if idx != -1:
            rel = path[idx + len('chexpert/'):]
            return os.path.join(CHEXPERT_ROOT, rel)
    if 'nih' in path.lower():
        idx = path.lower().find('data/')
        if idx != -1:
            rel = path[idx + len('data/'):]
            return os.path.join(NIH_ROOT, 'images', rel)
    return path

df['image_path_fixed'] = df['image_path'].apply(fix_image_path)

# ─────────────────────────────────────────────
# Verify a sample of images actually exist
# ─────────────────────────────────────────────
sample = df.sample(min(200, len(df)), random_state=42)
found = sample['image_path_fixed'].apply(os.path.exists).sum()
print(f'Image existence check: {found}/{len(sample)} found in sample')

if found < len(sample) * 0.5:
    print('WARNING: Less than 50% of images found!')
    print('Check that CheXpert and NIH datasets are added as Kaggle inputs.')
    print('Falling back to VQA-RAD images only...')

    # Fallback: use only VQA-RAD JSON with local images
    with open(VQARAD_JSON) as f:
        vqa_data = json.load(f)

    records = []
    for item in vqa_data:
        img_path = os.path.join(VQARAD_IMG_DIR, item['image_name'])
        if os.path.exists(img_path):
            records.append({
                'image_path_fixed': img_path,
                'question'        : item['question'],
                'answer'          : item['answer'],
                'answer_type'     : item['answer_type'].strip().lower(),
                'question_type'   : item['question_type'],
                'options'         : None,
                'split'           : 'train',
            })

    df_fallback = pd.DataFrame(records)
    # 80/10/10 split
    n = len(df_fallback)
    df_fallback.loc[:int(n*0.8), 'split'] = 'train'
    df_fallback.loc[int(n*0.8):int(n*0.9), 'split'] = 'val'
    df_fallback.loc[int(n*0.9):, 'split'] = 'test'
    df = df_fallback
    print(f'Using VQA-RAD fallback: {len(df)} pairs')
else:
    # Filter to only rows where image exists
    df = df[df['image_path_fixed'].apply(os.path.exists)].copy()
    df['answer_type'] = df['answer_type'].str.strip().str.lower()
    print(f'After filtering existing images: {len(df):,} rows')

print('\nFinal dataset distribution:')
print(df.groupby(['split', 'answer_type']).size().to_string())

## Cell 5: Build LLaVA-Med Prompt Templates

In [ ]:
def build_prompt(row: pd.Series) -> dict:
    """
    Convert a dataset row into the LLaVA-Med conversation format.

    LLaVA-Med uses the Mistral chat template:
    <s>[INST] <image>\n{question} [/INST] {answer} </s>

    For multiple choice, we include the options in the question.
    """
    question = str(row['question']).strip()
    answer   = str(row['answer']).strip().lower()

    # Append options for multiple choice
    if row['question_type'] == 'multiple_choice' and pd.notna(row.get('options')):
        opts = str(row['options'])
        # Options stored as 'A|B|C|D'
        opt_list = opts.split('|')
        labels = ['A', 'B', 'C', 'D', 'E']
        formatted_opts = '  '.join(
            f'({labels[i]}) {o.strip()}'
            for i, o in enumerate(opt_list)
            if i < len(labels)
        )
        question = f'{question}\nOptions: {formatted_opts}'

        # Find which option letter matches the answer
        for i, opt in enumerate(opt_list):
            if answer in opt.strip().lower() or opt.strip().lower() in answer:
                answer = f'({labels[i]}) {opt.strip()}'
                break

    # Add a clinical context hint for chest X-ray questions
    system_hint = 'You are a medical AI assistant specialized in chest X-ray analysis. '
    system_hint += 'Answer the following question based on the provided chest X-ray image.'

    return {
        'image_path' : row['image_path_fixed'],
        'question'   : question,
        'answer'     : answer,
        'answer_type': row['answer_type'],
        'system'     : system_hint,
        # Full prompt text (without image token — added during tokenization)
        'prompt'     : f'{system_hint}\n\nQuestion: {question}',
        'full_text'  : f'{system_hint}\n\nQuestion: {question}\nAnswer: {answer}',
    }


# Apply to all rows
print('Building prompts ...')
records = df.apply(build_prompt, axis=1).tolist()
print(f'Built {len(records):,} prompt records')

# Split
train_data = [r for r, s in zip(records, df['split']) if s == 'train']
val_data   = [r for r, s in zip(records, df['split']) if s == 'val']
test_data  = [r for r, s in zip(records, df['split']) if s == 'test']

# Apply sampling limits
if CFG['max_train_samples']:
    random.shuffle(train_data)
    train_data = train_data[:CFG['max_train_samples']]
if CFG['max_val_samples']:
    random.shuffle(val_data)
    val_data = val_data[:CFG['max_val_samples']]

print(f'Train: {len(train_data):,}  Val: {len(val_data):,}  Test: {len(test_data):,}')

# Show sample prompts
print('\n--- Sample CLOSED prompt ---')
closed = next((r for r in train_data if r['answer_type'] == 'closed'), None)
if closed:
    print(f'Q: {closed["question"]}')
    print(f'A: {closed["answer"]}')

print('\n--- Sample OPEN prompt ---')
opened = next((r for r in train_data if r['answer_type'] == 'open'), None)
if opened:
    print(f'Q: {opened["question"]}')
    print(f'A: {opened["answer"]}')

## Cell 6: Load LLaVA-Med Model & Tokenizer (4-bit quantized)

In [ ]:
from transformers import (
    LlavaForConditionalGeneration,
    AutoProcessor,
    BitsAndBytesConfig,
)

print(f'Loading model: {MODEL_ID}')
print('This downloads ~14GB on first run. Subsequent runs use cache.')

# 4-bit quantization config
# Reduces VRAM from ~28GB → ~7GB, enabling T4 fine-tuning
bnb_config = BitsAndBytesConfig(
    load_in_4bit               = True,
    bnb_4bit_quant_type        = 'nf4',        # NormalFloat4 — best quality
    bnb_4bit_compute_dtype     = torch.float16, # compute in fp16
    bnb_4bit_use_double_quant  = True,          # double quantization saves more VRAM
)

# Load processor (handles both image and text preprocessing)
processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)
processor.tokenizer.padding_side = 'right'

# Load model in 4-bit
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config = bnb_config,
    device_map          = 'auto',       # automatically places on GPU
    trust_remote_code   = True,
    torch_dtype         = torch.float16,
)

print('Model loaded.')
print(f'Model dtype: {model.dtype}')

# Count parameters
total    = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total/1e9:.2f}B')
print(f'Trainable parameters: {trainable/1e6:.2f}M  ({100*trainable/total:.3f}%)')

## Cell 7: Apply LoRA Adapters

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

# Step 1: Prepare model for k-bit (4-bit) training
# This casts LayerNorm to fp32 for stability and enables gradient checkpointing
model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True,
)

# Step 2: Define LoRA config
# We apply adapters to attention projection layers in the LLM
lora_config = LoraConfig(
    r              = CFG['lora_r'],
    lora_alpha     = CFG['lora_alpha'],
    lora_dropout   = CFG['lora_dropout'],
    target_modules = CFG['target_modules'],
    bias           = 'none',
    task_type      = 'CAUSAL_LM',
)

# Step 3: Wrap model with LoRA
model = get_peft_model(model, lora_config)

# Print updated parameter counts
model.print_trainable_parameters()

# Expected output:
# trainable params: 4,194,304 || all params: 7,067,861,504 || trainable%: 0.0594

## Cell 8: Custom Dataset Class

In [ ]:
from torch.utils.data import Dataset, DataLoader


class MedVQADataset(Dataset):
    """
    Loads (image, question, answer) triplets and formats them
    into the LLaVA-Med Mistral conversation template.

    Template:
      <s>[INST] <image>\n{system}\n\nQuestion: {question} [/INST]
      Answer: {answer} </s>

    The model learns to predict the answer tokens given
    the image + question prefix.
    """

    def __init__(self, records: list, processor, max_seq_length: int = 512):
        self.records        = records
        self.processor      = processor
        self.max_seq_length = max_seq_length

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx: int) -> dict:
        rec = self.records[idx]

        # ── Load image ──
        try:
            image = Image.open(rec['image_path']).convert('RGB')
            image = image.resize((CFG['image_size'], CFG['image_size']))
        except Exception:
            # If image fails to load, use a blank black image
            image = Image.new('RGB', (CFG['image_size'], CFG['image_size']), 0)

        # ── Build conversation text ──
        # LLaVA-Med Mistral format
        conversation = (
            f"[INST] <image>\n{rec['system']}\n\n"
            f"Question: {rec['question']} [/INST] "
            f"Answer: {rec['answer']}"
        )

        # ── Tokenize ──
        encoding = self.processor(
            text    = conversation,
            images  = image,
            return_tensors  = 'pt',
            padding         = 'max_length',
            truncation      = True,
            max_length      = self.max_seq_length,
        )

        input_ids      = encoding['input_ids'].squeeze(0)        # [seq_len]
        attention_mask = encoding['attention_mask'].squeeze(0)   # [seq_len]
        pixel_values   = encoding['pixel_values'].squeeze(0)     # [3, H, W]

        # ── Build labels ──
        # We only compute loss on the ANSWER tokens, not the prompt.
        # Find where [/INST] token ends, mask everything before as -100.
        labels = input_ids.clone()

        # Find the [/INST] boundary
        # Tokenize just the prompt (without answer) to find boundary
        prompt_only = (
            f"[INST] <image>\n{rec['system']}\n\n"
            f"Question: {rec['question']} [/INST] Answer: "
        )
        prompt_enc = self.processor(
            text   = prompt_only,
            images = image,
            return_tensors = 'pt',
            truncation     = True,
            max_length     = self.max_seq_length,
        )
        prompt_len = prompt_enc['input_ids'].shape[-1]

        # Mask prompt tokens — model should not learn to predict prompt
        labels[:prompt_len] = -100
        # Also mask padding
        labels[attention_mask == 0] = -100

        return {
            'input_ids'     : input_ids,
            'attention_mask': attention_mask,
            'pixel_values'  : pixel_values,
            'labels'        : labels,
        }


# Build datasets
print('Building train dataset ...')
train_dataset = MedVQADataset(train_data, processor, CFG['max_seq_length'])
print('Building val dataset ...')
val_dataset   = MedVQADataset(val_data,   processor, CFG['max_seq_length'])

print(f'Train: {len(train_dataset):,} samples')
print(f'Val:   {len(val_dataset):,} samples')

# Quick test
sample = train_dataset[0]
print('\nSample batch shapes:')
for k, v in sample.items():
    print(f'  {k}: {v.shape}  dtype={v.dtype}')

## Cell 9: Training Setup (Trainer)

In [ ]:
from transformers import TrainingArguments, Trainer
import math

steps_per_epoch = math.ceil(
    len(train_dataset) / (CFG['batch_size'] * CFG['grad_accum'])
)
total_steps = steps_per_epoch * CFG['epochs']
eval_steps  = max(50, steps_per_epoch // 2)   # evaluate ~2x per epoch
save_steps  = eval_steps

print(f'Steps per epoch : {steps_per_epoch}')
print(f'Total steps     : {total_steps}')
print(f'Eval every      : {eval_steps} steps')

training_args = TrainingArguments(
    output_dir                  = OUTPUT_DIR,

    # Batch
    per_device_train_batch_size = CFG['batch_size'],
    per_device_eval_batch_size  = CFG['batch_size'],
    gradient_accumulation_steps = CFG['grad_accum'],

    # Optimizer
    learning_rate               = CFG['learning_rate'],
    lr_scheduler_type           = 'cosine',
    warmup_ratio                = CFG['warmup_ratio'],
    weight_decay                = 0.01,
    optim                       = 'paged_adamw_8bit',   # 8-bit AdamW saves VRAM

    # Epochs
    num_train_epochs            = CFG['epochs'],

    # Precision
    fp16                        = True,
    bf16                        = False,

    # Logging
    logging_steps               = 20,
    logging_dir                 = f'{OUTPUT_DIR}/logs',
    report_to                   = 'none',

    # Evaluation & saving
    evaluation_strategy         = 'steps',
    eval_steps                  = eval_steps,
    save_strategy               = 'steps',
    save_steps                  = save_steps,
    save_total_limit            = 2,           # keep only 2 best checkpoints
    load_best_model_at_end      = True,
    metric_for_best_model       = 'eval_loss',
    greater_is_better           = False,

    # Memory
    gradient_checkpointing      = True,
    dataloader_pin_memory       = False,
    dataloader_num_workers      = 2,

    # Misc
    seed                        = CFG['seed'],
    remove_unused_columns       = False,       # important for LLaVA
)

print('TrainingArguments set.')

## Cell 10: Run Training

In [ ]:
trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = train_dataset,
    eval_dataset    = val_dataset,
    tokenizer       = processor.tokenizer,
)

print('='*60)
print('Starting training...')
print(f'  Training samples : {len(train_dataset):,}')
print(f'  Validation samples: {len(val_dataset):,}')
print(f'  Epochs           : {CFG["epochs"]}')
print(f'  Effective batch  : {CFG["batch_size"] * CFG["grad_accum"]}')
print(f'  Total steps      : {total_steps}')
print('='*60)

# Free VRAM before training
torch.cuda.empty_cache()

train_result = trainer.train()

print('\nTraining complete!')
print(f'  Final train loss: {train_result.training_loss:.4f}')
print(f'  Training runtime: {train_result.metrics["train_runtime"]:.0f}s')

## Cell 11: Save LoRA Adapter Weights

In [ ]:
print('Saving LoRA adapter weights...')

# Save only adapter weights (much smaller than full model)
model.save_pretrained(OUTPUT_DIR)
processor.save_pretrained(OUTPUT_DIR)

# Also save config for easy reloading
config_to_save = {
    'base_model'      : MODEL_ID,
    'lora_r'          : CFG['lora_r'],
    'lora_alpha'      : CFG['lora_alpha'],
    'target_modules'  : CFG['target_modules'],
    'training_samples': len(train_dataset),
    'epochs'          : CFG['epochs'],
    'final_train_loss': train_result.training_loss,
    'image_size'      : CFG['image_size'],
    'max_seq_length'  : CFG['max_seq_length'],
}

with open(f'{OUTPUT_DIR}/training_config.json', 'w') as f:
    json.dump(config_to_save, f, indent=2)

print(f'Model saved to: {OUTPUT_DIR}')

# List saved files
for fname in os.listdir(OUTPUT_DIR):
    size = os.path.getsize(f'{OUTPUT_DIR}/{fname}') / 1e6
    print(f'  {fname:40s}  {size:.1f} MB')

## Cell 12: Evaluation on Test Set

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from collections import defaultdict

print('Evaluating on test set (first 500 samples)...')

model.eval()
eval_samples = test_data[:500]

results = defaultdict(lambda: {'correct': 0, 'total': 0, 'bleu': []})

for i, rec in enumerate(eval_samples):
    if i % 50 == 0:
        print(f'  Processing {i}/{len(eval_samples)}...')

    try:
        image = Image.open(rec['image_path']).convert('RGB')
        image = image.resize((CFG['image_size'], CFG['image_size']))
    except Exception:
        continue

    prompt = (
        f"[INST] <image>\n{rec['system']}\n\n"
        f"Question: {rec['question']} [/INST] Answer:"
    )

    inputs = processor(
        text   = prompt,
        images = image,
        return_tensors = 'pt',
    ).to(DEVICE)

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens = 32,
            do_sample      = False,   # greedy decoding for determinism
            temperature    = 1.0,
            pad_token_id   = processor.tokenizer.eos_token_id,
        )

    # Decode only the new tokens (not the prompt)
    new_tokens = output_ids[0][inputs['input_ids'].shape[-1]:]
    predicted  = processor.tokenizer.decode(new_tokens, skip_special_tokens=True).strip().lower()
    ground_truth = rec['answer'].strip().lower()

    answer_type = rec['answer_type']
    results[answer_type]['total'] += 1

    if answer_type == 'closed':
        # Exact match or substring match
        if (ground_truth in predicted) or (predicted in ground_truth):
            results[answer_type]['correct'] += 1
    else:
        # BLEU score for open questions
        ref    = [ground_truth.split()]
        hyp    = predicted.split()
        smooth = SmoothingFunction().method1
        bleu   = sentence_bleu(ref, hyp, smoothing_function=smooth)
        results[answer_type]['bleu'].append(bleu)
        # Partial match for open
        if ground_truth in predicted or any(
            w in predicted for w in ground_truth.split() if len(w) > 3
        ):
            results[answer_type]['correct'] += 1

print('\n' + '='*50)
print('EVALUATION RESULTS')
print('='*50)

for atype, stats in results.items():
    if stats['total'] > 0:
        acc = 100 * stats['correct'] / stats['total']
        print(f'  {atype.upper():10s}  Accuracy: {acc:5.1f}%  (n={stats["total"]})')
        if stats['bleu']:
            avg_bleu = np.mean(stats['bleu'])
            print(f'             Avg BLEU: {avg_bleu:.4f}')

print('\nSample predictions:')
for rec in eval_samples[:3]:
    try:
        image = Image.open(rec['image_path']).convert('RGB').resize(
            (CFG['image_size'], CFG['image_size'])
        )
        prompt = (
            f"[INST] <image>\n{rec['system']}\n\n"
            f"Question: {rec['question']} [/INST] Answer:"
        )
        inputs = processor(text=prompt, images=image, return_tensors='pt').to(DEVICE)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=32, do_sample=False)
        new_toks = out[0][inputs['input_ids'].shape[-1]:]
        pred = processor.tokenizer.decode(new_toks, skip_special_tokens=True).strip()
        print(f'  Q: {rec["question"][:70]}')
        print(f'  Expected : {rec["answer"]}')
        print(f'  Predicted: {pred}')
        print()
    except Exception as e:
        print(f'  Sample failed: {e}')

## Cell 13: Zip & Download

In [ ]:
import zipfile

zip_path = '/kaggle/working/medvqa_adapter.zip'

print(f'Creating zip: {zip_path}')
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fname in os.listdir(OUTPUT_DIR):
        fpath = os.path.join(OUTPUT_DIR, fname)
        if os.path.isfile(fpath):
            zf.write(fpath, arcname=fname)
            size_mb = os.path.getsize(fpath) / 1e6
            print(f'  Added: {fname}  ({size_mb:.1f} MB)')

zip_size = os.path.getsize(zip_path) / 1e6
print(f'\nZip created: {zip_path}  ({zip_size:.1f} MB)')
print('\nTo download:')
print('  Kaggle sidebar → Output → medvqa_adapter.zip → Download')

## Training Complete!

**What to do next:**
1. Download `medvqa_adapter.zip` from Kaggle Output
2. Extract to `model/medvqa_adapter/` in your project
3. Restart backend — `/medvqa` endpoint will be ready

**Expected file sizes:**
- `adapter_model.bin` → ~80 MB (just the LoRA weights)
- `adapter_config.json` → ~1 KB
- `tokenizer files` → ~3 MB
- **Total zip: ~85 MB** (not 14GB — only adapters saved, not full model)